In [ ]:
!pip install requests folium pandas geopandas -q 

##  Análise de ilícitos ambientais - Ibama
![](https://wikilai.fiquemsabendo.com.br/images/thumb/a/a8/Logo_principal_-_fiquem_sabendo.png/320px-Logo_principal_-_fiquem_sabendo.png)

In [ ]:
#@title Importar GeoPandas
import geopandas as gpd
import pandas as pd
import requests
import numpy as np
import requests
import io
import folium
from folium.plugins import HeatMap


# Importação dos Dados Ibama e TSE

In [ ]:
#@title GeoJson dos Embargos do Ibama maxfeatures = 100000
url_embargos_ibama ='https://siscom.ibama.gov.br/geoserver/publica/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=publica:vw_brasil_adm_embargo_a&maxFeatures=100000&outputFormat=application%2Fjson'


In [ ]:
#@title Ler GeoJson como GeoDataFrame
# Ler GeoDataFrame
# Adicionar um cabeçalho User-Agent para evitar o erro 403 Forbidden
# A URL do IBAMA pode bloquear requisições sem um User-Agent ou com User-Agents genéricos.

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

# Fazer a requisição HTTP usando requests
response = requests.get(url_embargos_ibama, headers=headers)
response.raise_for_status() # Lança uma exceção para erros HTTP (4xx ou 5xx)

# Usar io.BytesIO para tratar o conteúdo da resposta como um arquivo para o geopandas
gdf = gpd.read_file(io.BytesIO(response.content))

# Converter colunas para minúscula
gdf.columns = gdf.columns.str.lower()

In [ ]:
gdf.shape

# Importação via leitura de arquivo geojson

In [ ]:
#gdf2 = gpd.read_file("C:/Users/Administrador/Documents/GitHub/Ibama/dados/gpkg_file.geojson")

In [ ]:
gdf['cpf_cnpj_infrator'].value_counts()

## DADOS ELEIÇÃO

In [ ]:
import requests, zipfile, io

# Exemplo: candidatos de 2022
url = "https://cdn.tse.jus.br/estatistica/sead/odsele/consulta_cand/consulta_cand_2022.zip"

# Baixar e abrir o zip
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

# Ver arquivos disponíveis
print(z.namelist())

# Ler um CSV específico
df = pd.read_csv(z.open("consulta_cand_2022_BRASIL.csv"), sep=";", encoding="latin1")



In [ ]:
df.columns

In [ ]:
gdf.columns

In [ ]:
gdf = gdf.rename(columns={'cpf_cnpj_infrator':'NR_CPF_CANDIDATO'})

In [ ]:
gdf['NR_CPF_CANDIDATO'].value_counts()

In [ ]:
df['NR_CPF_CANDIDATO'].value_counts()

## REMOVER CNPJs

In [ ]:
padrao_cnpj = r"\d{2}\.\d{3}\.\d{3}/\d{4}-\d{2}"

gdf_cpfs = gdf[~gdf["NR_CPF_CANDIDATO"].str.match(padrao_cnpj, na=False)]
gdf_cpfs.shape


In [ ]:
gdf_cpfs['NR_CPF_CANDIDATO'].value_counts()

In [ ]:
df["tamanho"] = df["NR_CPF_CANDIDATO"].astype(str).str.len()
display(df[["NR_CPF_CANDIDATO","tamanho"]])

In [ ]:
# Mantém apenas linhas com 11 dígitos
df_filtrado = df[df["tamanho"] == 11]
df_filtrado.shape

In [19]:
df_filtrado['NR_CPF_CANDIDATO'].value_counts()

NR_CPF_CANDIDATO
26043947315    3
61028347200    3
87042967372    3
20315503300    2
86362720172    2
              ..
55852831204    1
20041373200    1
34771956200    1
52318265100    1
59294825000    1
Name: count, Length: 19575, dtype: int64

In [20]:
gdf_cpfs.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 78189 entries, 1 to 89661
Data columns (total 29 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   id                    78189 non-null  str                
 1   nom_pessoa            72773 non-null  str                
 2   NR_CPF_CANDIDATO      69204 non-null  str                
 3   seq_tad               78189 non-null  str                
 4   numero_tad            78189 non-null  str                
 5   data_tad              78188 non-null  datetime64[ms]     
 6   num_latitude_tad      66434 non-null  str                
 7   num_longitude_tad     65577 non-null  str                
 8   processo_tad          72204 non-null  str                
 9   sig_uf                78189 non-null  str                
 10  nom_municipio         78189 non-null  str                
 11  num_auto_infracao     65321 non-null  str                
 12  s

In [21]:
repetidos = gdf_cpfs[gdf_cpfs["NR_CPF_CANDIDATO"].isin(df_filtrado["NR_CPF_CANDIDATO"])]
print("Repetidos encontrados:")
print(repetidos)

Repetidos encontrados:
Empty GeoDataFrame
Columns: [id, nom_pessoa, NR_CPF_CANDIDATO, seq_tad, numero_tad, data_tad, num_latitude_tad, num_longitude_tad, processo_tad, sig_uf, nom_municipio, num_auto_infracao, ser_auto_infracao, cod_municipio, cod_uf, status_tad, qtd_area_desmatada, data_cadastro_tad, des_infracao, serie_tad, sit_embarga_poligono, legislacao, artigo_legislacao, artigo, imagem_validacao, respeita_embargo, data_geom, orgao, geometry]
Index: []

[0 rows x 29 columns]


## TRATAMENTO DE DADOS PARA MAIS DE UMA ELEIÇÂO